In [1]:
import os

# set GPU where the code should run
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig
from huggingface_hub import login
import torch
import pandas as pd
import gc
import re

QUANTIZE_4BIT=True
USE_FLASH_ATTENTION=False
access_token = ''

In [ ]:
#load dataset
df = pd.read_json('500_samples_DATA_removedTextsLarger8319characters.json', lines=True)
df= df.iloc[800:]
df.head()

In [4]:
# define prompt templates: 3 (in already generated data, id =2), 9, 10
templates = [
    {
        "id":3
        ,
        "system_prompt": """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X. 
X steht für 'andere'.""",
        "prompt": """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens}"""
    },
    {
        "id":9,
        "system_prompt": """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
""",
        "prompt": """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes. 
Eingabesatz: {tokens}"""
    },
    {
        "id":10,
        "system_prompt": """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
""",
        "prompt": """"Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes zurück. 
Hier sind ein paar Beispiele, was Sie machen sollen:
Eingabesatz: ['Balde', 'rings', 'die', 'Sterne', 'bleichen', 'Und', 'ermattet', 'von', 'Beschwerde', 'Wächsern', 'ihre', 'Wangen', 'bleichen', '.', 'Fäulnis', 'wittert', 'aus', 'der', 'Erde', '.']
Ausgabe: [('Balde','NN'),('rings','ADV'),('die','ART'),('Sterne','NN'),('bleichen','VVINF'),('Und','KON'),('ermattet','VVFIN'),('von','APPR'),('Beschwerde','NN'),('Wächsern','KOUS'),('ihre','PPOSAT'),('Wangen','NN'),('bleichen','VVINF'),('.','$.'),('Fäulnis','NN'),('wittert','VVFIN'),('aus','APPR'),('der','ART'),('Erde','NN'),('.','$.')]

Eingabesatz: ['Traurig', 'rauscht', 'das', 'Rohr', 'im', 'Tümpel', 'Und', 'sie', 'friert', 'in', 'sich', 'gekauert', '.', 'Fern', 'ein', 'Hahn', 'kräht', '.', 'Übern', 'Tümpel', 'Hart', 'und', 'grau', 'der', 'Morgen', 'schauert', '.']
Ausgabe: [('Traurig','ADJD'), ('rauscht','VVFIN'), ('das','ART'), ('Rohr','NN'), ('im','APPRART'), ('Tümpel','NN'), ('Und','KON'), ('sie','PPER'), ('friert','VVFIN'), ('in','APPR'), ('sich','PRF'), ('gekauert','VVPP'), ('.','$.'), ('Fern','NN'),('ein','ART'), ('Hahn','NN'), ('kräht','VVFIN'), ('.','$.'), ('Übern','NN'), ('Tümpel','NN'), ('Hart','NE'), ('und','KON'), ('grau','ADJD'), ('der','ART'), ('Morgen','NN'), ('schauert','VVFIN'), ('.','$.')]

Eingabesatz: ['In', 'der', 'Schmiede', 'dröhnt', 'der', 'Hammer', 'Und', 'sie', 'huscht', 'am', 'Tor', 'vorüber', '.', 'Glührot', 'schwingt', 'der', 'Knecht', 'den', 'Hammer', 'Und', 'sie', 'schaut', 'wie', 'tot', 'hinüber', '.']
Ausgabe: [('In','APPR'), ('der','ART'), ('Schmiede','NN'), ('dröhnt','VVFIN'), ('der','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('huscht','VVFIN'), ('am','APPART'), ('Tor','NN'), ('vorüber','PTKVZ'), ('.','$.'), ('Glührot','NN'), ('schwingt','VVFIN'), ('der','ART'), ('Knecht','NN'), ('den','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('schaut','VVFIN'), ('wie','KOKOM'), ('tot','ADJD'), ('hinüber','ADV'), ('.','$.')]
Eingabesatz: {tokens}"""
    }
]


In [5]:
#login on hugginface to use model
#login()

In [ ]:
#load model 
model_id = "CohereForAI/aya-23-8B"

quantization_config = None
if QUANTIZE_4BIT:
  quantization_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_use_double_quant=True,
      bnb_4bit_compute_dtype=torch.bfloat16,
  )

attn_implementation = None
if USE_FLASH_ATTENTION:
  attn_implementation="flash_attention_2"

model = AutoModelForCausalLM.from_pretrained(
          model_id,
          token=access_token,
          quantization_config=quantization_config,
          attn_implementation=attn_implementation,
          torch_dtype=torch.bfloat16,
          device_map={"": 0},
        )

In [ ]:
#load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [8]:
#method to generate pos tags with model
def generate_pos_tags(prompts, model,tokenizer, temperature=0.3, top_p=0.95, max_new_tokens=2048):

  input_ids = tokenizer.apply_chat_template(
        prompts,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,
        truncation=True,
        return_tensors="pt",
      )
  input_ids = input_ids.to(model.device)
  prompt_padded_len = len(input_ids[0])

  gen_tokens = model.generate(
        input_ids,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        do_sample=True,
      )

  # get only generated tokens
  gen_tokens = [gt[prompt_padded_len:] for gt in gen_tokens]

  gen_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
  return gen_text

In [ ]:
#list to store results
results = []

#number of rows to check in which bucket we are
id = 0

# Counter for chunks, if you need chunks
#chunk_counter = 7
#chunk_size = 400
    
# # POS-Tagging for every text in the three Prompt-Templates
for idx, row in df.iterrows():
        text = row['tokens']
        bucket = row['bucket']
        labels = row['labels']
        
        # for every Prompt-Template
        for i, template in enumerate(templates):
            # Structure prompts
            prompts = [
                {"role": "system", "content": template['system_prompt']},
                {"role": "user", "content": template['prompt'].format(tokens=text)}
            ]
            
            # generate pos tags
            pos_result = generate_pos_tags(prompts, model, tokenizer)
            
            # add 1 to id for new row
            id += 1
            
            # store result
            results.append({
                'year': row['year'],
                'bucket': bucket,
                'labels': labels,
                'template_id': template['id'], 
                'POS_Result': pos_result
            })
        
            # Check if we have reached 400 rows (or every 400 POS results) if you need chunks
            #if id % chunk_size == 0:
                # Convert results to DataFrame
                #df_results = pd.DataFrame(results)

                # Save the chunk to a JSON file
                #filename = f'500samples_RESULTS_aya23_chunk_{chunk_counter}.json'
                #df_results.to_json(filename, orient='records', lines=True)

                #print(f"Saved chunk {chunk_counter} to {filename}")

                 # Reset the results list for the next chunk
                #results = []
                #gc.collect                
                #torch.cuda.empty_cache()

                # Increment the chunk counter
                #chunk_counter += 1

# Save results, if chunk is used then Save remaining results if any rows are left after the loop
if results:
    df_results = pd.DataFrame(results)
    filename = f'500samples_RESULTS_aya23.json'
    df_results.to_json(filename, orient='records', lines=True)
    print(f"Saved final dataset to {filename}")
    
    #filename_chunk = f'500samples_RESULTS_aya23_chunk_{chunk_counter}.json'
    #df_results.to_json(filename_chunk, orient='records', lines=True)
    #print(f"Saved final chunk {chunk_counter} to {filename}")
            

In [23]:
#combine the chunks if used chunks
#chunk1 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_1.json', lines=True)
#chunk2 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_2.json', lines=True)
#chunk3 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_3.json', lines=True)
#chunk4 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_4.json', lines=True)
#chunk5 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_5.json', lines=True)
#chunk6 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_6.json', lines=True)
#chunk7 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_7.json', lines=True)
#chunk8 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_8.json', lines=True)
#chunk9 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_9.json', lines=True)
#chunk10 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_10.json', lines=True)
#chunk11 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_11.json', lines=True)
#chunk12 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_12.json', lines=True)
#chunk13 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_13.json', lines=True)
#chunk14 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_14.json', lines=True)
#chunk15 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_15.json', lines=True)
#chunk16 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_16.json', lines=True)
#chunk17 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_17.json', lines=True)
#chunk18 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_18.json', lines=True)
#chunk19 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_19.json', lines=True)
#chunk20 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_20.json', lines=True)
#chunk21 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_21.json', lines=True)
#chunk22 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_22.json', lines=True)
#chunk23 = pd.read_json('datasets/500samples_RESULTS_aya23_chunk_23.json', lines=True)

#df_list = [chunk1,chunk2,chunk3,chunk4,chunk5,chunk6,chunk7,chunk8,chunk9,chunk10,chunk11,chunk12,chunk13,chunk14,chunk15,chunk16,chunk17,
           #chunk18,chunk19,chunk20,chunk21,chunk22,chunk23]

# Concatenate all DataFrames into one
#df_results = pd.concat(df_list, axis=0)


# Reset index if needed
#df_results.reset_index(drop=True, inplace=True)
#df_results.head()

# Save the combined DataFrame into a new CSV
#df_results.to_json("500samples_RESULTS_aya23.json", orient='records', lines=True)

In [24]:
# function to extract POS-Tags with Regex
def extract_pos_tags(text):
    return re.findall(r"'([A-Z$]+(?:[A-Z.,(]+)?)'", text)

#apply regex
df_results['extracted_pos_tags'] = df_results['POS_Result'].apply(extract_pos_tags)

# store as json
df_results.to_json('500samples_RESULTS_aya23_extracted.json', orient='records', lines=True)

In [ ]:
#calculate average accuracy
def calculate_accuracy(pos_tags, predicted_tags):
    correct = sum(1 for pos, pred in zip(pos_tags, predicted_tags) if pos == pred)
    return correct / len(pos_tags) *100

average_accuracies = []
for bucket in df_results['bucket'].unique():
    for template_num in df_results['template_id'].unique():
        bucket_template_data = df_results[(df_results['bucket'] == bucket) & (df_results['template_id'] == template_num)]
        
        # calculate average accuracy for this bucket and this template
        accuracies = []
        for idx, row in bucket_template_data.iterrows():
            extracted_tags = row['extracted_pos_tags']
            true_tags = row['labels'] 
            accuracy = calculate_accuracy(true_tags, extracted_tags)
            accuracies.append(accuracy)
        
        average_accuracy = sum(accuracies) / len(accuracies)  if accuracies else 0
        
        average_accuracies.append({
            'bucket': bucket,
            'template': template_num,
            'average_accuracy': average_accuracy
        })

# store accuracy results as DataFrame
df_accuracy = pd.DataFrame(average_accuracies)       
# print head of dataframe
print(df_accuracy.head())

# store accuracy results as JSON
df_accuracy.to_json('500samples_ACCURACY_aya23.json', orient='records', lines=True)

In [ ]:
# print finish script
print("Script finished successfully!")

# Beende das Skript
import os
os.system("exit")